In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path
import os

# --- 1. Define File Paths ---
h5ad_dir = Path("fetal_brain_hongtaoyu/0318_fetralbrain_rawcount_bin100")
csv_dir = Path("fetal_brain_hongtaoyu/bin100_label")

output_base_dir = Path("Process_Data/bin100_3D_region")

output_base_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- 2. Find all h5ad files ---
h5ad_files = list(h5ad_dir.glob("*.h5ad"))

if not h5ad_files:
    print(f"Error: No .h5ad files found in {h5ad_dir}.")
    exit()

print(f"Found {len(h5ad_files)} .h5ad files in total.Starting processing...")
print("-" * 30)

In [ ]:
# --- 3. Loop through each file ---
for h5ad_file_path in h5ad_files:
    file_name = h5ad_file_path.name  # e.g., "9_D03466F1G2_WT202403310042.h5ad"
    file_stem = h5ad_file_path.stem  # e.g., "9_D03466F1G2_WT202403310042"
    
    csv_file_name = file_stem + ".csv"
    csv_file_path = csv_dir / csv_file_name
    
    print(f"\n>>> Processing file: {file_name}")

    # --- 3.1 Check if the corresponding CSV file exists ---
    if not csv_file_path.exists():
        print(f"  [Warning] Corresponding CSV label file not found: {csv_file_path}. Skipping this file.")
        continue

    try:
        # --- 3.2 Read h5ad and csv files ---
        adata = sc.read_h5ad(h5ad_file_path)
        adata_obs = pd.read_csv(csv_file_path, index_col=0)

        if adata_obs.index.dtype=='int64':
            adata_obs.index = adata_obs.index.astype(str)
        if adata.obs_names.dtype=='int64':
            adata.obs_names = adata.obs_names.astype(str)
        # --- 3.3 Align and merge data (based on your provided logic) ---
        
        # To be safe, we only keep indices (barcodes) common to both files
        common_indices = adata.obs.index.intersection(adata_obs.index)
        
        if len(common_indices) == 0:
            print(f"  [Warning] No common barcodes between {file_name} and {csv_file_name}. Skipping this file.")
            continue
            
        if len(common_indices) < len(adata_obs):
             print(f"  [Note] Some barcodes in the CSV file are not in the h5ad file.")
        
        # 1. Filter adata based on the indices from the CSV
        adata = adata[common_indices, :].copy() 
        # 2. Ensure adata_obs also only contains the common indices (in the same order)
        adata_obs_filtered = adata_obs.loc[common_indices]
        # 3. Replace obs
        adata.obs = adata_obs_filtered

        # --- 3.4 Check for 'region_h2' column ---
        if 'region_h2' not in adata.obs.columns:
            print(f"  [Warning] Column 'region_h2' not found in {csv_file_name}. Skipping this file.")
            continue

        # --- 3.5 Split by region and save ---
        print(f"  Splitting file by 'region_h2'...")
        
        # Get all unique region names
        unique_regions = adata.obs['region_h2'].unique()
        
        # Print the spot count for each region (as requested)
        print("  [Spot Statistics]:")
        for region in unique_regions:
            if pd.isna(region):
                count = adata.obs['region_h2'].isna().sum()
                print(f"    - 'NaN' (Undefined Region): {count} spots")
            else:
                count = (adata.obs['region_h2'] == region).sum()
                print(f"    - '{region}': {count} spots")

        # Loop through each region, create a sub-adata, and save it
        for region in unique_regions:
            # Handle NaN values, saving them to an "NA" folder or skipping
            if pd.isna(region):
                region_name_str = "NA" # Or you could 'continue' to skip
                print(f"    Processing 'NaN' region, will save to '{region_name_str}' folder...")
                adata_region = adata[adata.obs['region_h2'].isna()].copy()
            else:
                region_name_str = str(region)
                adata_region = adata[adata.obs['region_h2'] == region_name_str].copy()
            
            # 1. Create the output directory for this specific region
            #    e.g., "Process_Data/bin100_3D_region/vz"
            region_output_dir = output_base_dir / region_name_str
            region_output_dir.mkdir(parents=True, exist_ok=True)
            
            # 2. Define the final output file path
            #    e.g., "Process_Data/bin100_3D_region/vz/9_D03466F1G2_WT202403310042.h5ad"
            output_h5ad_path = region_output_dir / file_name
            
            # 3. Save the subsetted h5ad file
            if adata_region.n_obs > 0:
                adata_region.write_h5ad(output_h5ad_path)
                # print(f"    Saved: {output_h5ad_path}") # (Uncomment for more detailed logging)
            else:
                print(f"    Region '{region_name_str}' has 0 spots. Not saving empty file.")
                
        print(f"  <<< Finished processing file: {file_name}")

    except Exception as e:
        print(f"  [!! ERROR !!] An error occurred while processing {file_name}: {e}")
        continue

print("-" * 30)
print("All files processed.")

In [ ]:
# --- 3. Loop through each file ---
h5ad_files = ['A03994F1G2_WT2024071215067.h5ad','116_B03610D5E6_WT202403310078.h5ad']
for h5ad_file_path in h5ad_files:
    file_name = h5ad_file_path  # e.g., "9_D03466F1G2_WT202403310042.h5ad"
    file_stem = h5ad_file_path.split('.')[0]  # e.g., "9_D03466F1G2_WT202403310042"
    
    csv_file_name = file_stem + ".csv"
    csv_file_path = csv_dir / csv_file_name
    h5ad_file_path = h5ad_dir/ h5ad_file_path
    print(f"\n>>> Processing file: {file_name}")

    # --- 3.1 Check if the corresponding CSV file exists ---
    if not csv_file_path.exists():
        print(f"  [Warning] Corresponding CSV label file not found: {csv_file_path}. Skipping this file.")
        continue

    try:
        # --- 3.2 Read h5ad and csv files ---
        adata = sc.read_h5ad(h5ad_file_path)
        adata_obs = pd.read_csv(csv_file_path, index_col=0)

        if adata_obs.index.dtype=='int64':
            adata_obs.index = adata_obs.index.astype(str)
        if adata.obs_names.dtype=='int64':
            adata.obs_names = adata.obs_names.astype(str)
        # --- 3.3 Align and merge data (based on your provided logic) ---
        
        # To be safe, we only keep indices (barcodes) common to both files
        common_indices = adata.obs.index.intersection(adata_obs.index)
        
        if len(common_indices) == 0:
            print(f"  [Warning] No common barcodes between {file_name} and {csv_file_name}. Skipping this file.")
            continue
            
        if len(common_indices) < len(adata_obs):
             print(f"  [Note] Some barcodes in the CSV file are not in the h5ad file.")
        
        # 1. Filter adata based on the indices from the CSV
        adata = adata[common_indices, :].copy() 
        # 2. Ensure adata_obs also only contains the common indices (in the same order)
        adata_obs_filtered = adata_obs.loc[common_indices]
        # 3. Replace obs
        adata.obs = adata_obs_filtered

        # --- 3.4 Check for 'region_h2' column ---
        if 'region_h2' not in adata.obs.columns:
            print(f"  [Warning] Column 'region_h2' not found in {csv_file_name}. Skipping this file.")
            continue

        # --- 3.5 Split by region and save ---
        print(f"  Splitting file by 'region_h2'...")
        
        # Get all unique region names
        unique_regions = adata.obs['region_h2'].unique()
        
        # Print the spot count for each region (as requested)
        print("  [Spot Statistics]:")
        for region in unique_regions:
            if pd.isna(region):
                count = adata.obs['region_h2'].isna().sum()
                print(f"    - 'NaN' (Undefined Region): {count} spots")
            else:
                count = (adata.obs['region_h2'] == region).sum()
                print(f"    - '{region}': {count} spots")

        # Loop through each region, create a sub-adata, and save it
        for region in unique_regions:
            # Handle NaN values, saving them to an "NA" folder or skipping
            if pd.isna(region):
                region_name_str = "NA" # Or you could 'continue' to skip
                print(f"    Processing 'NaN' region, will save to '{region_name_str}' folder...")
                adata_region = adata[adata.obs['region_h2'].isna()].copy()
            else:
                region_name_str = str(region)
                adata_region = adata[adata.obs['region_h2'] == region_name_str].copy()
            
            # 1. Create the output directory for this specific region
            #    e.g., "Process_Data/bin100_3D_region/vz"
            region_output_dir = output_base_dir / region_name_str
            region_output_dir.mkdir(parents=True, exist_ok=True)
            
            # 2. Define the final output file path
            #    e.g., "Process_Data/bin100_3D_region/vz/9_D03466F1G2_WT202403310042.h5ad"
            output_h5ad_path = region_output_dir / file_name
            
            # 3. Save the subsetted h5ad file
            if adata_region.n_obs > 0:
                adata_region.write_h5ad(output_h5ad_path)
                # print(f"    Saved: {output_h5ad_path}") # (Uncomment for more detailed logging)
            else:
                print(f"    Region '{region_name_str}' has 0 spots. Not saving empty file.")
                
        print(f"  <<< Finished processing file: {file_name}")

    except Exception as e:
        print(f"  [!! ERROR !!] An error occurred while processing {file_name}: {e}")
        continue

print("-" * 30)
print("All files processed.")

# ChP

In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path
import os

# --- 1. Define File Paths ---
h5ad_dir = Path("fetal_brain_data_ChP")

output_base_dir = Path("Process_Data/bin100_3D_region")

output_base_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- 2. Find all h5ad files ---
h5ad_files = list(h5ad_dir.glob("*.h5ad"))

if not h5ad_files:
    print(f"Error: No .h5ad files found in {h5ad_dir}.")
    exit()

print(f"Found {len(h5ad_files)} .h5ad files in total.Starting processing...")
print("-" * 30)

In [ ]:
# --- 3. Loop through each file ---
for h5ad_file_path in h5ad_files:
    file_name = h5ad_file_path.name  # e.g., "9_D03466F1G2_WT202403310042.h5ad"
    file_stem = h5ad_file_path.stem  # e.g., "9_D03466F1G2_WT202403310042"
    
    print(f"\n>>> Processing file: {file_name}")

    try:
        # --- 3.2 Read h5ad and csv files ---
        adata = sc.read_h5ad(h5ad_file_path)

        if 'region_h2' not in adata.obs.columns:
            print(f"  [Warning] Column 'region_h2' not found in {csv_file_name}. Skipping this file.")
            continue

        # --- 3.5 Filter for 'cb_chp' region and save ---
        print(f"  Filtering file for 'region_h2' == 'cb_chp'...")
        
        target_region = 'cb_chp'
        
        # Get all unique region names (still useful for stats)
        unique_regions = adata.obs['region_h2'].unique()
        
        # Print the spot count for each region (as requested)
        print("  [Spot Statistics]:")
        for region in unique_regions:
            if pd.isna(region):
                count = adata.obs['region_h2'].isna().sum()
                print(f"    - 'NaN' (Undefined Region): {count} spots")
            else:
                count = (adata.obs['region_h2'] == region).sum()
                print(f"    - '{region}': {count} spots")

        # --- Check if 'cb_chp' exists and process it ---
        if target_region not in unique_regions:
            print(f"    [Note] Target region '{target_region}' not found in this file. Skipping save.")
        else:
            # Region exists, now filter and save
            print(f"    Found '{target_region}'. Filtering and saving...")
            
            adata_region = adata[adata.obs['region_h2'] == target_region].copy()
            
            # 1. Create the output directory
            #    e.g., "Process_Data/bin100_3D_region/cb_chp"
            region_output_dir = output_base_dir / target_region
            region_output_dir.mkdir(parents=True, exist_ok=True)
            
            # 2. Define the final output file path
            #    e.g., ".../cb_chp/9_D03466F1G2_WT202403310042.h5ad"
            output_h5ad_path = region_output_dir / file_name
            
            # 3. Save the subsetted h5ad file
            if adata_region.n_obs > 0:
                adata_region.write_h5ad(output_h5ad_path)
                # print(f"    Saved: {output_h5ad_path}") # (Uncomment for more detailed logging)
            else:
                print(f"    Region '{target_region}' has 0 spots. Not saving empty file.")
                
        print(f"  <<< Finished processing file: {file_name}")

    except Exception as e:
        print(f"  [!! ERROR !!] An error occurred while processing {file_name}: {e}")
        continue

print("-" * 30)
print("All files processed.")